<a href="https://colab.research.google.com/github/SoyebAhmed5/flyrank-internship-ml/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SoyebAhmed5/flyrank-internship-ml/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

# Research Question

## Can content metadata help prioritize pages for human review when identifying potentially declining content?

### Problem Statement

Content teams manage large numbers of published pages. Some pages may experience declining search performance, but manually reviewing every page requires time and resources.

This project investigates whether selected content metadata can help prioritize pages for further investigation.

The objective is not to automatically decide which pages should be refreshed. Instead, the goal is to develop an exploratory decision-support workflow that ranks content items and provides understandable reasons for human review.

### Research Objectives

1. Examine patterns associated with growing and declining content.
2. Develop a baseline approach for prioritizing content.
3. Train and evaluate a Logistic Regression model.
4. Compare the model against the baseline using the same validation split.
5. Audit possible data leakage and evaluate generalization using client-grouped validation.
6. Convert model outputs into a human-reviewed content action playbook.

### Research Question

Can a Logistic Regression model using content metadata provide useful prioritization signals beyond a simple rule-based baseline when evaluated on previously unseen clients?

### Scope

This is an exploratory machine learning study. Its results are intended to support human review and prioritization, not automated content changes or guaranteed predictions of future search performance.

In [3]:
import pandas as pd
from pathlib import Path

# Load the anonymized starter dataset
DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    DATA_URL = (
        "https://raw.githubusercontent.com/"
        "SoyebAhmed5/flyrank-internship-ml/main/"
        "data/raw/content_refresh_anonymized.csv"
    )
    df = pd.read_csv(DATA_URL)

# Basic dataset information
print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))

# Verify important fields
required_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "content_age_days"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("\nUnique clients:", df["client_id"].nunique())

print("\nTrend direction distribution:")
print(df["trend_direction"].value_counts(dropna=False))

print("\nMissing values in important columns:")
print(df[required_columns].isnull().sum())

print("\nDataset inspection completed.")

Dataset shape: (30000, 44)
Number of columns: 44

Unique clients: 32

Trend direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Missing values in important columns:
content_id             0
client_id              0
trend_direction        0
trend_pct           3388
content_age_days       0
dtype: int64

Dataset inspection completed.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

## Data Sources and Scope

### Starter Dataset

The model experiment uses the anonymized content dataset:

`data/raw/content_refresh_anonymized.csv`

The dataset contains approximately **30,000 content records and 44 columns**, representing content from 32 anonymized clients. Each row represents one content item.

The dataset includes content metadata, search-performance measurements, engagement metrics, and trend-related fields. Several performance fields represent trailing 90-day measurements.

### Warehouse Context

The broader FlyRank warehouse release is `v20260703`. Its content-performance fact table contains approximately 78.8 million records, covering 27 January 2025 through 30 June 2026.

The warehouse and starter CSV must be treated as separate data scopes. The Logistic Regression experiment described in this notebook uses the 30,000-row starter dataset; the warehouse size must not be presented as the classifier's training-data size.

### Exclusions and Data Handling

The following fields are excluded from model features:

- `content_id` and `client_id`: pseudonymous identifiers, used only for record identification and grouped validation.
- `trend_direction` and `trend_pct`: directly related to the target label and therefore potential sources of target leakage.
- Search-performance and engagement aggregates: excluded because their measurement windows may overlap with the outcome period or contain information unavailable at prediction time.

The target is derived from `trend_direction`, with declining content represented by the `down` category.

The dataset does not provide a sufficiently documented reporting timestamp for a defensible chronological split. Therefore, the model experiment uses client-grouped validation to evaluate performance on clients not represented in the training partition.

### Privacy

The analysis uses anonymized records. Client names, content URLs, private search queries, and raw content identifiers are not included in the public paper or published result artifacts.

In [4]:
# Dataset dimensions
total_records = len(df)
total_features = len(df.columns)
unique_clients = df["client_id"].nunique()

# Target distribution
label_distribution = (
    df["trend_direction"]
    .value_counts(dropna=False)
    .rename_axis("trend_direction")
    .reset_index(name="count")
)

print("DATASET SUMMARY")
print("----------------")
print("Total records:", total_records)
print("Total columns:", total_features)
print("Unique clients:", unique_clients)

print("\nLABEL DISTRIBUTION")
display(label_distribution)

# Verify that the target categories are understood
print("\nAvailable trend categories:")
print(df["trend_direction"].dropna().unique())

DATASET SUMMARY
----------------
Total records: 30000
Total columns: 44
Unique clients: 32

LABEL DISTRIBUTION


,trend_direction,count
0,down,16262
1,stable,5962
2,up,4388
3,new,2236
4,flat,1152



Available trend categories:
['down' 'stable' 'new' 'up' 'flat']


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## Methodology

### Prediction Objective

The task is framed as a binary classification problem. The objective is to identify content records labelled as declining so that content teams can prioritize pages for human review.

### Target Label

The target variable is derived from `trend_direction`:

- `1`: The content item is labelled `down`.
- `0`: The content item has another trend category.

The model therefore estimates the likelihood of the dataset's declining label. It does not establish that a page will decline in the future.

### Feature Selection

The experiment uses the following content metadata:

**Numerical features**
- `content_age_days`
- `search_volume`
- `competition`
- `cpc`
- `word_count`
- `char_count`

**Categorical features**
- `content_type`
- `main_intent`
- `competition_level`

Identifiers such as `content_id` and `client_id` are excluded from model inputs. Label-related fields and performance aggregates are also excluded to reduce the risk of target leakage and information-window overlap.

### Preprocessing

Numerical missing values are replaced using the median. Categorical missing values are replaced using the most frequent category. Numerical features are standardized, and categorical features are one-hot encoded.

Preprocessing is fitted on the training partition through a machine-learning pipeline so that evaluation data does not determine training transformations.

### Model

Logistic Regression is used as the classification model. It provides a relatively interpretable baseline for evaluating whether the selected metadata contains useful predictive signals.

### Validation Design

The data is divided using an 80/20 client-grouped split with `GroupShuffleSplit` and random state 42. All records belonging to a client are assigned to either training or evaluation, never both.

This tests generalization to clients not represented in the training partition. It is not a chronological validation design because the dataset does not provide a sufficiently documented reporting timestamp.

### Baseline Comparison

The Logistic Regression model must be compared with the rule-based baseline developed earlier in the internship. Both approaches must be evaluated on the same held-out clients and against the same target labels.

The baseline's exact rule and evaluation results will be included after verifying the ML-07 implementation.

### Leakage Checks

The experiment excludes the target field, its related trend-percentage field, pseudonymous identifiers as predictors, and performance aggregates that may overlap with the outcome measurement window.

Client overlap between training and evaluation partitions is checked explicitly. These safeguards reduce known leakage risks, but they do not prove that every possible source of temporal or measurement leakage has been eliminated.

In [5]:
# Import required libraries
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    accuracy_score
)

# Define numerical and categorical features
numeric_features = [
    "content_age_days",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count"
]

categorical_features = [
    "content_type",
    "main_intent",
    "competition_level"
]

feature_columns = numeric_features + categorical_features

# Prepare modelling dataset
model_df = df.dropna(
    subset=["trend_direction", "client_id"]
).copy()

# Binary target: down = 1, all other categories = 0
y = (model_df["trend_direction"] == "down").astype(int)

# Features and client groups
X = model_df[feature_columns].copy()
groups = model_df["client_id"]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Positive class rate:", round(y.mean(), 4))

print("\nSelected features:")
print(feature_columns)

Feature matrix shape: (30000, 9)
Target shape: (30000,)
Positive class rate: 0.5421

Selected features:
['content_age_days', 'search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_type', 'main_intent', 'competition_level']


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [6]:
# Create an 80/20 client-grouped split
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    group_splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

# Verify client separation
train_clients = set(groups_train.unique())
test_clients = set(groups_test.unique())

client_overlap = train_clients.intersection(test_clients)

print("TRAINING PARTITION")
print("------------------")
print("Rows:", len(X_train))
print("Clients:", len(train_clients))
print("Positive rate:", round(y_train.mean(), 4))

print("\nEVALUATION PARTITION")
print("--------------------")
print("Rows:", len(X_test))
print("Clients:", len(test_clients))
print("Positive rate:", round(y_test.mean(), 4))

print("\nClient overlap:", len(client_overlap))
assert len(client_overlap) == 0, "Client leakage detected!"

TRAINING PARTITION
------------------
Rows: 23837
Clients: 25
Positive rate: 0.5501

EVALUATION PARTITION
--------------------
Rows: 6163
Clients: 7
Positive rate: 0.511

Client overlap: 0


## 5. Limitations

*What this work cannot claim.*

In [7]:
# Numerical preprocessing
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Complete model pipeline
group_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train model
group_model.fit(X_train, y_train)

# Generate evaluation predictions
group_prob = group_model.predict_proba(X_test)[:, 1]
group_pred = (group_prob >= 0.5).astype(int)

print("Logistic Regression training completed.")
print("Evaluation predictions generated:", len(group_pred))

Logistic Regression training completed.
Evaluation predictions generated: 6163


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [8]:
# Calculate evaluation metrics
model_metrics = {
    "Average Precision": average_precision_score(y_test, group_prob),
    "ROC-AUC": roc_auc_score(y_test, group_prob),
    "Accuracy": accuracy_score(y_test, group_pred),
    "Precision": precision_score(y_test, group_pred, zero_division=0),
    "Recall": recall_score(y_test, group_pred, zero_division=0),
    "F1 Score": f1_score(y_test, group_pred, zero_division=0)
}

print("LOGISTIC REGRESSION — GROUPED EVALUATION")
print("----------------------------------------")

for metric, value in model_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nEvaluation base rate:", round(y_test.mean(), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, group_pred))

LOGISTIC REGRESSION — GROUPED EVALUATION
----------------------------------------
Average Precision: 0.5422
ROC-AUC: 0.5478
Accuracy: 0.5327
Precision: 0.5359
Recall: 0.6380
F1 Score: 0.5825

Evaluation base rate: 0.511

Confusion Matrix:
[[1274 1740]
 [1140 2009]]


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [9]:
# Calculate evaluation metrics
model_metrics = {
    "Average Precision": average_precision_score(y_test, group_prob),
    "ROC-AUC": roc_auc_score(y_test, group_prob),
    "Accuracy": accuracy_score(y_test, group_pred),
    "Precision": precision_score(y_test, group_pred, zero_division=0),
    "Recall": recall_score(y_test, group_pred, zero_division=0),
    "F1 Score": f1_score(y_test, group_pred, zero_division=0)
}

print("LOGISTIC REGRESSION — GROUPED EVALUATION")
print("----------------------------------------")

for metric, value in model_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nEvaluation base rate:", round(y_test.mean(), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, group_pred))

LOGISTIC REGRESSION — GROUPED EVALUATION
----------------------------------------
Average Precision: 0.5422
ROC-AUC: 0.5478
Accuracy: 0.5327
Precision: 0.5359
Recall: 0.6380
F1 Score: 0.5825

Evaluation base rate: 0.511

Confusion Matrix:
[[1274 1740]
 [1140 2009]]


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
